# Phase 1: SQL Analytics Layer
## Healthcare AI System

**Objective:** Load raw hospital CSVs into a relational SQLite database and run operational + financial analytics queries.

**Tables:**
- `patients` - 5,000 rows
- `visits`   - 25,000 rows
- `billing`  - 25,000 rows

In [1]:
# Inputs
import pandas as pd
import sqlite3
import os

In [3]:
# Load raw CSV Files
patients = pd.read_csv("../data/patients.csv")
visits = pd.read_csv('../data/visits.csv')
billing = pd.read_csv('../data/billing.csv')

print("patients shape:", patients.shape)
print("visits shape:", visits.shape)
print("billing shape:", billing.shape)

patients shape: (5000, 7)
visits shape: (25000, 8)
billing shape: (25000, 7)


In [4]:
# Create Sqlite db
os.makedirs("../db", exist_ok=True)
db_path = "../db/healthcare.db"

# Connect to sqlite and create db
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print("Database connected: ", conn)

Database connected:  <sqlite3.Connection object at 0x0000023F5BAB6F20>


In [5]:
# Load dataframes into sqlite tables
# write all three dataframes into sqlite tables
patients.to_sql("patients", conn, index=False, if_exists="replace")
visits.to_sql("visits", conn, index=False, if_exists="replace")
billing.to_sql("billing", conn, index=False, if_exists="replace")

print("All three tables have been created/loaded")

All three tables have been created/loaded


In [7]:
# Quick preview of tables
print("=== PATIENTS (first 3 rows) ===")
print(pd.read_sql("SELECT * FROM patients limit 3", conn).to_string())

print("=== VISITS (first 3 rows) ===")
print(pd.read_sql("SELECT * FROM visits limit 3", conn).to_string())

print("=== BILLING (first 3 rows) ===")
print(pd.read_sql("SELECT * FROM billing limit 3", conn).to_string())

=== PATIENTS (first 3 rows) ===
   patient_id  age gender       city insurance_provider  chronic_flag registration_date
0           1   53      M  Hyderabad         SecureLife             0        2025-05-14
1           2   42      M       Pune         HealthPlus             0        2025-11-18
2           3   56      F  Hyderabad         HealthPlus             0        2025-05-11
=== VISITS (first 3 rows) ===
   visit_id  patient_id  visit_date   department visit_type  length_of_stay_hours risk_score  doctor_id
0         1         756  2025-10-18   Cardiology         ER                  3.48        Low        169
1         2        4102  2025-04-06  Orthopedics        OPD                 15.31       High        148
2         3        2964  2025-07-13          ICU         ER                 34.36        Low        153
=== BILLING (first 3 rows) ===
   bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
0        1         1       23577.37           

In [11]:
dept_workload = pd.read_sql("""
SELECT department,
    COUNT(visit_id)                                         as total_visits,
    ROUND(AVG(length_of_stay_hours), 2)                     as average_los_hours,
    ROUND(MAX(length_of_stay_hours), 2)                     as max_los_hours,
    SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END)    as hight_risk_visits
FROM visits
GROUP BY department
ORDER BY total_visits DESC
""", conn)

print(dept_workload.to_string(index=False))

 department  total_visits  average_los_hours  max_los_hours  hight_risk_visits
    General          4228              19.43          78.42                839
         ER          4220              19.53          74.81                872
  Neurology          4165              19.72          70.12                846
Orthopedics          4164              19.66          71.60                842
 Cardiology          4159              19.60          77.42                790
        ICU          4064              19.36          72.47                845


In [13]:
# Doctor Risk Cases
# Which doctors handle the most high-risk patients?
doctor_risk = pd.read_sql("""
select
    doctor_id,
    count(visit_id) as total_visits,
    SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) as high_risk_visits,
    ROUND(100.0 * SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) / count(visit_id), 1) as high_risk_pct
FROM visits
GROUP BY doctor_id
ORDER BY high_risk_visits DESC
LIMIT 10
""", conn)

print("Top 10 Doctors by High Risk Cases")
print(doctor_risk.to_string(index=False))

Top 10 Doctors by High Risk Cases
 doctor_id  total_visits  high_risk_visits  high_risk_pct
       174           264                71           26.9
       198           252                69           27.4
       169           279                68           24.4
       177           266                67           25.2
       135           261                65           24.9
       105           250                65           26.0
       188           285                64           22.5
       180           290                64           22.1
       131           266                62           23.3
       178           245                61           24.9


In [16]:
# Patient Visit Patterns
# How many visits does each patient make on average?
visit_patterns = pd.read_sql("""
SELECT
    visit_frequency_bucket,
    COUNT(patient_id) as num_patients
FROM (
    SELECT
        patient_id,
        COUNT(visit_id) as visit_count,
        CASE
            when count(visit_id) = 1 then '1 visit'
            when count(visit_id) between 2 and 3 then '2-3 visits'
            when count(visit_id) between 4 and 5 then '4-5 visits'
            else '6+ visits'
        END as visit_frequency_bucket
    FROM visits
    GROUP BY patient_id
)
GROUP BY visit_frequency_bucket
ORDER BY num_patients DESC
""", conn)

print("Patient Visit Frequency Distribution")
print(visit_patterns.to_string(index=False))

Patient Visit Frequency Distribution
visit_frequency_bucket  num_patients
             6+ visits          1937
            4-5 visits          1731
            2-3 visits          1150
               1 visit           149


In [17]:
# Risk Score Distribution by Department
risk_by_dept = pd.read_sql("""
select
    department,
    sum(case when risk_score = 'Low' then 1 else 0 end) as low,
    sum(case when risk_score = 'Medium' then 1 else 0 end) as medium,
    sum(case when risk_score = 'High' then 1 else 0 end) as high,
    count(*) as total
from visits
group by department
order by high desc
""", conn)

print("Risk Score Distribution by Department")
print(risk_by_dept.to_string(index=False))

Risk Score Distribution by Department
 department  low  medium  high  total
         ER 2092    1256   872   4220
  Neurology 2058    1261   846   4165
        ICU 2037    1182   845   4064
Orthopedics 2078    1244   842   4164
    General 2123    1266   839   4228
 Cardiology 2082    1287   790   4159


# Financial Analytics

In [21]:
# Insurance Billing Breakdown
# Revenu and claim outcomes by insurance provided.
insurance_billing = pd.read_sql("""
select
    p.insurance_provider,
    count(b.bill_id) as total_claims,
    round(sum(b.billed_amount), 0) as total_billed,
    round(avg(b.billed_amount), 0) as avg_billed,
    round(sum(b.approved_amount), 0) as total_approved,
    sum(case when b.claim_status = 'Paid' then 1 else 0 end) as paid,
    sum(case when b.claim_status = 'Pending' then 1 else 0 end) as pending,
    sum(case when b.claim_status = 'Rejected' then 1 else 0 end) as rejected
from billing b
    join visits v on b.visit_id = v.visit_id
    join patients p on v.patient_id = p.patient_id
    group by p.insurance_provider
    order by total_billed DESC
""", conn)

print("Insurance Billing Distribution")
print(insurance_billing.to_string(index=False))

Insurance Billing Distribution
insurance_provider  total_claims  total_billed  avg_billed  total_approved  paid  pending  rejected
         MediCareX          6532   134591163.0     20605.0     100135469.0  3875     1661       996
           CareOne          6283   130707993.0     20803.0      96997758.0  3787     1562       934
        HealthPlus          6220   130180741.0     20929.0      96251775.0  3680     1609       931
        SecureLife          5965   126289040.0     21172.0      93770886.0  3598     1431       936


In [22]:
# Claim Rejection Analysis
# Which insurance providers reject the most claims?
rejection_analysis = pd.read_sql("""
select
    p.insurance_provider,
    count(b.bill_id) as total_claims,
    sum(case when b.claim_status = 'Rejected' then 1 else 0 end) as rejected_claims,
    round(100.0 * sum(case when b.claim_status = 'Rejected' then 1 else 0 end) / count(b.bill_id), 1) as rejection_rate
from billing b
    join visits v on b.visit_id = v.visit_id
    join patients p on v.patient_id = p.patient_id
    group by p.insurance_provider
    order by rejection_rate DESC
""", conn)
print("Rejection Analysis")
print(rejection_analysis.to_string(index=False))

Rejection Analysis
insurance_provider  total_claims  rejected_claims  rejection_rate
        SecureLife          5965              936            15.7
         MediCareX          6532              996            15.2
        HealthPlus          6220              931            15.0
           CareOne          6283              934            14.9


In [24]:
# Revenue Realization
# How much of what we bill actually gets approved?
revenue_realization = pd.read_sql("""
select
    p.insurance_provider,
    round(sum(b.billed_amount), 0) as total_billed,
    round(sum(b.approved_amount), 0) as total_approved,
    round(100.0 * sum(b.approved_amount) / sum(b.billed_amount), 1) as realization_rate_pct
from billing b
    join visits v on b.visit_id = v.visit_id
    join patients p on v.patient_id = p.patient_id
    group by p.insurance_provider
    order by realization_rate_pct DESC
""", conn)
print("Revenue Analysis")
print(revenue_realization.to_string(index=False))

Revenue Analysis
insurance_provider  total_billed  total_approved  realization_rate_pct
         MediCareX   134591163.0     100135469.0                  74.4
        SecureLife   126289040.0      93770886.0                  74.3
           CareOne   130707993.0      96997758.0                  74.2
        HealthPlus   130180741.0      96251775.0                  73.9


# Data Quality

In [25]:
# Missing Records Check
print("=== MISSING VALUES CHECK ===\n")

for table in ['patients', 'visits', 'billing']:
    df = pd.read_sql(f"select * from {table}", conn)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"{table}")
        print(nulls.to_string())
        print()
    else:
        print(f"{table}: No missing values\n")

=== MISSING VALUES CHECK ===

patients: No missing values

visits: No missing values

billing
approved_amount    1318
payment_days        790



In [26]:
# Duplicated Patients Check
duplicate_patients = pd.read_sql("""
select
    p.patient_id,
    count(*) as num_duplicates
from patients p
group by p.patient_id
having count(*) > 1
""", conn)

if len(duplicate_patients) == 0:
    print("No duplicate patient_ids found")
else:
    print(f"Found {len(duplicate_patients)} duplicate patient_ids")
    print(duplicate_patients)

No duplicate patient_ids found


In [29]:
# Orphan Records Check
# Visits without matching patients
orphan_visits = pd.read_sql("""
select count(*) as orphan_visits
from visits v
    left join patients p on v.patient_id = p.patient_id
    where p.patient_id is null
""", conn)

# Billing without matching visits
orphan_billing = pd.read_sql("""
select count(*) as orphan_billing
from billing b
    left join visits v on b.visit_id = v.visit_id
    where v.visit_id is null
""", conn)

print("Orphan visits (no matching patients)", orphan_visits['orphan_visits'][0])
print("Orphan billing (no matching visits)", orphan_billing['orphan_billing'][0])

Orphan visits (no matching patients) 0
Orphan billing (no matching visits) 0


In [30]:
# Invalid LOS Check
invalid_los = pd.read_sql("""
select
      count(case when length_of_stay_hours <= 0 then 1 end) as zero_or_negative,
      count(case when length_of_stay_hours > 720 then 1 end) as over_30_days,
      min(length_of_stay_hours) as min_los,
      max(length_of_stay_hours) as max_los,
      round(avg(length_of_stay_hours), 2) as avg_los
from visits
""", conn)

print("LOS Validation:")
print(invalid_los.to_string(index=False))

LOS Validation:
 zero_or_negative  over_30_days  min_los  max_los  avg_los
                0             0      0.5    78.42    19.55


In [31]:
# Invalid Payment Days Check
invalid_payment = pd.read_sql("""
select
     count(case when payment_days < 0 then 1 end) as negative_days,
     count(case when payment_days > 365 then 1 end) as over_1_year,
     count(case when payment_days is null then 1 end) as nulls,
     min(payment_days) as min_days,
     max(payment_days) as max_days
from billing
""", conn)
print("Payment Validation:")
print(invalid_payment.to_string(index=False))

Payment Validation:
 negative_days  over_1_year  nulls  min_days  max_days
             0            0    790       1.0      55.0


In [32]:
# Join all three tables into one flat table for ML modeling.
# This is the single source of truth for Phase 2 EDA and Phase 3 Modeling.
model_table = pd.read_sql("""
select
    -- Patient features
    p.patient_id,
    p.age,
    p.gender,
    p.city,
    p.insurance_provider,
    p.chronic_flag,
    p.registration_date,

    -- Visit features
    v.visit_id,
    v.visit_date,
    v.department,
    v.visit_type,
    v.length_of_stay_hours,
    v.risk_score,
    v.doctor_id,

    -- Billing features
    b.bill_id,
    b.billed_amount,
    b.approved_amount,
    b.claim_status,
    b.payment_days,
    b.billing_date
from visits v
join patients p on v.patient_id = p.patient_id
join billing b on v.visit_id = b.visit_id
order by v.visit_date
""", conn)

print("model_table shape:", model_table.shape)
print("Columns:", model_table.columns.tolist())
model_table.head(3)

model_table shape: (25000, 20)
Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,3993,9,M,Hyderabad,HealthPlus,1,2025-10-18,684,2025-01-20,General,ICU,5.79,Low,137,684,36721.68,36721.68,Paid,19.0,2025-08-15
1,76,59,M,Delhi,HealthPlus,0,2025-07-09,1412,2025-01-20,General,ER,34.80,Medium,109,1412,8365.47,4189.20,Pending,26.0,2025-11-14
2,3393,43,M,Hyderabad,HealthPlus,0,2025-07-10,1510,2025-01-20,ICU,ICU,31.37,Medium,135,1510,16529.35,0.00,Rejected,7.0,2025-04-24


In [33]:
# Save the table
import os
os.makedirs("../outputs", exist_ok=True)
model_table.to_csv("../outputs/model_table.csv", index=False)
print(f"model_table.csv saved --> /outputs/model_table.csv")
print(f"Shape: {model_table.shape}")
print(f"Size: {os.path.getsize('../outputs/model_table.csv')/1024:.1f} KB")

model_table.csv saved --> /outputs/model_table.csv
Shape: (25000, 20)
Size: 3126.7 KB


In [ ]:
# Close Connection
conn.close()
print("Database connection closed.")